## Scalability Testing

A stress test: 4 GCSs monitoring 60 vehicles each — 240 in total — every one
flying a small AUTO square. It exercises the framework at a scale the other
notebooks don't reach.

At this vehicle count **`NoVisualizer` is the only feasible backend**: Gazebo
tops out around 3 UAVs and QGroundControl around 25. The `Gazebo` and
`QGroundControl` cells below are kept only for parity with the other
notebooks — the `Simulator` is handed `novis`, and Remote ID is switched off
(`Oracle(rid_enabled=False)`) to keep the run light.

In [ ]:
from simulator.config import Color, Model
from simulator.entities import SimGCS, SimVehicle
from simulator.helpers import SimProcess, clean
from simulator.helpers.coordinates import ENUPose, GRAPose
from simulator.oracle import Oracle
from simulator.planner import AutoPlan, Plan
from simulator.sim import Simulator
from simulator.visualizer import NoVisualizer

clean()

## Simulation Positions

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

gcs_colors = [Color.RED, Color.GREEN, Color.BLUE, Color.YELLOW]
n_uavs_per_gcs = 60
side_len = 10
altitude = 5
model = Model.IRIS
base_homes = ENUPose.list(
    [
        (i * 50, j * 3 * side_len, 0, 0)
        for i in range(len(gcs_colors))
        for j in range(n_uavs_per_gcs)
    ]
)


## Create Vehicles and GCSs

In [ ]:
sysids = range(1, len(base_homes) + 1)
colors: list[Color] = []
gcss: dict[Color, SimGCS] = {}
for color in gcs_colors:
    colors.extend([color] * n_uavs_per_gcs)
    gcss[color] = SimGCS(name=f"{color.name}_{color.emoji}")

vehs: list[SimVehicle] = []
for sysid, base_home, color in zip(sysids, base_homes, colors, strict=True):
    auto_plan = AutoPlan.square_traj(
        side_len=side_len,
        alt=altitude,
        name="simple_auto_plan",
        sysid=sysid,
        gra_origin=gra_origin,
        relative_home=base_home,
        firmware=model.firmware,
    )

    veh = SimVehicle.from_relative(
        sysid=sysid,
        gcss=[gcss[color]],
        plan=auto_plan,
        color=color,
        enu_origin=enu_origin,
        relative_home=base_home,
        relative_path=Plan.create_square_path(side_len=side_len),
        model=model,
    )
    vehs.append(veh)


In [ ]:
for gcs in gcss.values():
    print(gcs)

## No Visualizer

In [ ]:
novis = NoVisualizer(gra_origin)


## Oracle and Simulator

In [ ]:
orac = Oracle(rid_enabled=False)

for veh in vehs:
    orac.add_vehicle(veh)

simulator = Simulator(
    oracle=orac,
    visualizer=novis,
    terminals=[SimProcess.GCS],
    verbose=1,
)


## Run

In [ ]:
simulator.run()


## Plot some of the recorded trajectories

In [ ]:
orac.plot_trajectories(
    legend=False,
    xlim=(0, 160),
    sysids=[i for i in range(1, 5)]
    + [i + 60 for i in range(1, 5)]
    + [i + 120 for i in range(1, 5)]
    + [i + 180 for i in range(1, 5)],
);